# 02 · Requisitos — Hit Maker (Spotify)

Módulo anterior: [01 · Guiding Questions](01_guiding_questions.ipynb). Próximo módulo: [03 · Tratamento e Limpeza dos Dados](03_tratamento_limpeza_dados.ipynb).

Este módulo traduz as Guiding Questions em requisitos concretos — de dados, funcionais, não-funcionais e éticos — que orientam a implementação dos módulos seguintes. Não há execução de pipeline aqui, apenas o levantamento do estado real dos dados disponíveis no repositório e das decisões em aberto.

## Requisito de dados: quais fontes existem e para que servem

`archive/` tem 4 CSVs. Só um tem o schema completo exigido pelas funções do pipeline (colunas de `COLUNAS` + `FEATURES_NUMERICAS`):

| Arquivo | Linhas | Colunas-chave | Tem features de áudio? | Tem data de lançamento? | Uso pretendido |
|---|---|---|---|---|---|
| `archive/dataset.csv` | 114.000 | `track_genre`, `popularity`, `danceability` ... `speechiness` | ✅ Sim | ❌ Não | **Dataset principal** — treino/EDA (usado nos módulos 03–05) |
| `archive/Spotify Most Streamed Songs.csv` | 953 | `streams`, `bpm`, `danceability_%` ... (escala 0–100) | ⚠️ Parcial (nomes/escalas diferentes) | ✅ Sim (1930–2023) | Candidato a validação temporal — exige remapeamento de schema |
| `archive/spotify_data clean.csv` | 8.582 | metadados de artista/álbum (`artist_popularity`, `artist_followers`, `album_release_date`) | ❌ Não | ✅ Sim | Enriquecimento (feats, popularidade do artista) — não usável sozinho |
| `archive/track_data_final.csv` | 8.778 | mesmo padrão do anterior + `track_duration_ms` | ❌ Não | ✅ Sim | Enriquecimento — idem |

**Conclusão:** hoje só `dataset.csv` tem o schema completo. Os outros três respondem a GQs específicas do refinamento (seção "Base de Dados" de `docs/README.md`, tags `[MELHORADA]`/`[NOVA]`) mas exigem integração ainda não implementada:
- Enriquecer `dataset.csv` com `artist_popularity`/`artist_followers`/colaborações via os dois arquivos de metadados (join por `track_id` ou por artista+faixa).
- Padronizar `Spotify Most Streamed Songs.csv` (`danceability_%` 0–100 → 0–1, `streams` como proxy de popularidade) para servir de conjunto de validação temporal.

Essas duas integrações ficam como requisito **pendente**, fora do escopo do protótipo atual (módulos 03–05 usam só `dataset.csv`).

## Requisitos funcionais (por frente)

### Frente 1 — Auditoria e limpeza
- RF1.1 Carregar o dataset bruto (`carregar_dataset`).
- RF1.2 Reportar shape, nº de gêneros, % de nulos por coluna, duplicatas — linha inteira e por artista+faixa (`auditoria_dataset`).
- RF1.3 Visualizar a distribuição de faixas por gênero (`distribuicao_generos`).
- RF1.4 Detectar outliers por IQR nas features numéricas (`detectar_outliers_iqr`).
- RF1.5 Tratar nulos por estratégia configurável — mediana ou remoção (`tratar_valores_ausentes`).

### Frente 2 — EDA / Anatomia do sucesso
- RF2.1 Rotular faixas como hit/não-hit por percentil de `popularity`, global ou por gênero (`definir_hit`, `definir_hit_por_genero`).
- RF2.2 Comparar médias de features entre hits e não-hits (`comparar_hits_vs_demais`).
- RF2.3 Rankear gêneros por densidade de hits e volatilidade de popularidade (`densidade_e_volatilidade_por_genero`).

### Frente 3 — Modelagem preditiva
- RF3.1 Montar matriz de features (numéricas + one-hot de gênero) (`preparar_features`).
- RF3.2 Treinar classificador balanceado por classe (`treinar_modelo`).
- RF3.3 Avaliar com classification report, ROC-AUC, matriz de confusão, curva ROC (`avaliar_modelo`).
- RF3.4 Rankear features por importância (`importancia_features`).
- RF3.5 Segmentar faixas por perfil sonoro via KMeans, com apoio de método do cotovelo/silhouette (`escalar_features`, `encontrar_k_ideal`, `clusterizar_faixas`, `perfil_dos_clusters`, `visualizar_clusters_2d`).

### Frente 4 — Produto
- RF4.1 Construir perfil de referência (features médias dos hits) por gênero (`criar_perfil_referencia`).
- RF4.2 Comparar uma faixa específica a esse perfil, feature a feature (`comparar_musica_com_perfil`).
- RF4.3 Listar faixas de baixa popularidade com alta probabilidade prevista de hit (`identificar_faixas_subestimadas`).
- RF4.4 Listar faixas de baixa popularidade estruturalmente próximas (KNN) dos hits do mesmo gênero (`musicas_similares_a_hits`).

## Decisões de negócio em aberto

Estas são decisões que **precisam de validação do time** antes de qualquer versão além de protótipo — todas já sinalizadas como `# ATENÇÃO:` no código reaproveitado:

1. **Threshold de "hit"** — percentil 80 fixo é o ponto mais crítico do projeto. Um corte global penaliza gêneros de nicho que nunca atingem popularidade absoluta alta; a alternativa por gênero (`definir_hit_por_genero`) já existe no código mas não é a usada no pipeline principal.
2. **Nomes reais de colunas** — o dicionário `COLUNAS` assume o schema de `dataset.csv`; qualquer troca de fonte de dados exige revalidar esses nomes num só lugar.
3. **Tratamento de nulos** — imputar (mediana), remover, ou preservar? Decisão de negócio, não só técnica.
4. **Duplicidade real** — duplicata "de linha inteira" vs. duplicata por `(artista, faixa)`; qual conta para deduplicação?
5. **Encoding de gênero** — one-hot sobre ~114 categorias gera matriz esparsa; considerar target/frequency encoding ou agrupamento em macro-gêneros.
6. **Desbalanceamento de classes** — hit é ~20% dos dados por construção; `class_weight="balanced"` é o mínimo, SMOTE é a alternativa se a performance for baixa.
7. **Tamanho mínimo de amostra por gênero** — perfis de referência com menos de 30 hits no gênero são estatisticamente instáveis (aviso já implementado, mas o limite de 30 é arbitrário).
8. **Risco de vazamento de dados (data leakage)** — `identificar_faixas_subestimadas` precisa rodar sobre dados fora do treino do modelo, nunca sobre o mesmo split usado para treinar (limitação ainda presente na demo do módulo 05).
9. **Momento de uso do modelo** — pré-lançamento (sem `popularity` disponível) ou pós-lançamento? Muda quais features existem no momento da predição real.
10. **k do KMeans** — escolher olhando o gráfico de cotovelo/silhouette, não usar k fixo sem validar.

## Requisitos éticos (levantados, não implementados)

Da seção "Ética" de `docs/README.md` — nenhum destes está refletido no código ainda:

- Auditar viés de representação entre gêneros/culturas (over/under-representação sistemática).
- Confirmar licenciamento do dataset Kaggle e os Termos de Serviço da API do Spotify para uso do projeto (acadêmico).
- Mitigar risco de profecia autorrealizável — nunca usar o score como critério único/automático de decisão.
- Comunicar ao usuário o risco de redução de diversidade musical se artistas otimizarem para o "perfil de hit".
- Definir política de uso responsável (ferramenta de apoio, não decisão automática).
- Garantir explicabilidade do score na interface do produto (feature importance / SHAP), não só um número.

Ficam como requisito para o módulo de produto e para qualquer decisão de disponibilizar a ferramenta além de protótipo interno.

## Critério de "pronto" por módulo

- **03 (Limpeza):** dataset auditado, nulos/outliers/duplicatas tratados e documentados, dataframe limpo salvo em `artifacts/`.
- **04 (Ideação):** hit definido e justificado, EDA comparativa feita, modelo treinado e avaliado (métricas registradas), clusters caracterizados.
- **05 (Protótipo):** funções de produto demonstradas ponta a ponta com pelo menos 1 gênero real, output no formato pensado para o usuário final (ainda que não seja uma UI).